# Generación de Algoritmos Sintéticos

**Objetivo:** Generar variaciones sintéticas de algoritmos para crear datasets de entrenamiento y validación
**Duración estimada:** 40 minutos

---

## Contenido

1. [Setup](#setup)
2. [Arquitectura del Generador](#arquitectura-del-generador)
3. [Plantillas de Algoritmos](#plantillas-de-algoritmos)
4. [Generación de Variaciones](#generacion-de-variaciones)
5. [Generación por Complejidad Objetivo](#generacion-por-complejidad)
6. [Validación de Algoritmos Generados](#validacion)
7. [Estadísticas del Dataset Generado](#estadisticas)
8. [Exportación](#exportacion)

---

## 1. Setup

In [ ]:
import sys
import random
import json
from pathlib import Path

sys.path.insert(0, '../..')

from dataset_generator import (
    AlgorithmGenerator,
    GenerationConfig,
    generate_algorithm_variations,
    SyntheticDataCreator,
)
from app.core.parser import parse_pseudocode
from app.core.analyzer import AnalyzerEngine

random.seed(42)
print("Setup completado")

---

## 2. Arquitectura del Generador

El módulo `dataset_generator` proporciona un sistema para crear variaciones programáticas de algoritmos.
Esto es útil para:

- Entrenar y evaluar detectores de patrones
- Crear conjuntos de prueba con distribución controlada
- Generar casos de prueba que cubran casos límite específicos

### Componentes Principales

| Componente | Función |
|------------|---------|
| `AlgorithmGenerator` | Genera variaciones de plantillas existentes |
| `ComplexityLabeler` | Asigna etiquetas de complejidad automáticamente |
| `SyntheticDataCreator` | Crea datasets balanceados por categoría |
| `DatasetExporter` | Exporta el dataset en múltiples formatos |

In [ ]:
# Verificar que el módulo está disponible
try:
    generator = AlgorithmGenerator()
    creator = SyntheticDataCreator()
    print("Módulo de generación de datasets disponible")
    print(f"Plantillas disponibles: {generator.get_available_templates()}")
except Exception as e:
    print(f"Error al inicializar: {e}")
    print("Usando generación manual como alternativa")

---

## 3. Plantillas de Algoritmos

Las plantillas son la base del generador. Cada plantilla define la estructura de un tipo de algoritmo
con parámetros variables que permiten crear múltiples variaciones.

In [ ]:
# Plantillas manuales para demostración
PLANTILLAS = {
    "ciclo_simple": {
        "descripcion": "Algoritmo con un solo ciclo for",
        "big_o": "O(n)",
        "patron": "BRUTE_FORCE",
        "template": """
algorithm {nombre}({params})
begin
    {init}
    for i <- 1 to n do
        {cuerpo}
    end
    return {retorno}
end
""",
        "variables": {
            "nombres": ["suma", "conteo", "buscar", "recorrer", "calcular"],
            "params": ["A[], n", "n", "A[], n, x"],
            "init": ["total <- 0", "count <- 0", "resultado <- 0"],
            "cuerpo": [
                "total <- total + A[i]",
                "if (A[i] > 0) then\n        count <- count + 1\n    end",
                "resultado <- resultado + i"
            ],
            "retorno": ["total", "count", "resultado"]
        }
    },
    "ciclos_anidados": {
        "descripcion": "Algoritmo con dos ciclos for anidados",
        "big_o": "O(n^2)",
        "patron": "BRUTE_FORCE",
        "template": """
algorithm {nombre}(A[], n)
begin
    for i <- 1 to n do
        for j <- 1 to n do
            {cuerpo}
        end
    end
end
""",
        "variables": {
            "nombres": ["matrizSuma", "comparar", "encontrarPar", "validar"],
            "cuerpo": [
                "A[i][j] <- 0",
                "if (A[i] > A[j]) then\n            temp <- A[i]\n        end",
                "total <- total + A[i] + A[j]"
            ]
        }
    },
    "recursion_simple": {
        "descripcion": "Algoritmo recursivo con un caso base",
        "big_o": "O(n)",
        "patron": "RECURSIVE",
        "template": """
algorithm {nombre}(n)
begin
    if (n <= {base}) then
        return {val_base}
    end
    return {expresion_recursiva}
end
""",
        "variables": {
            "nombres": ["factorial", "suma_rec", "potencia", "cuenta_rec"],
            "base": ["0", "1"],
            "val_base": ["1", "0", "n"],
            "expresion_recursiva": [
                "n * {nombre}(n - 1)",
                "{nombre}(n - 1) + n",
                "n + {nombre}(n - 1)"
            ]
        }
    },
}

for nombre, plantilla in PLANTILLAS.items():
    print(f"Plantilla '{nombre}': BigO={plantilla['big_o']}, Patrón={plantilla['patron']}")

---

## 4. Generación de Variaciones

In [ ]:
def generar_variaciones_manual(plantilla_key, n_variaciones=5):
    """
    Genera variaciones de una plantilla seleccionando valores aleatorios.
    """
    plantilla = PLANTILLAS[plantilla_key]
    variaciones = []
    variables = plantilla["variables"]
    
    for i in range(n_variaciones):
        codigo = plantilla["template"]
        
        # Sustituir variables aleatorias
        nombre = random.choice(variables.get("nombres", ["algoritmo"]))
        codigo = codigo.replace("{nombre}", nombre)
        
        for clave, opciones in variables.items():
            if clave == "nombres":
                continue
            if isinstance(opciones, list) and opciones:
                valor = random.choice(opciones)
                # Sustituir referencia recursiva si existe
                valor = valor.replace("{nombre}", nombre)
                codigo = codigo.replace(f"{{{clave}}}", valor)
        
        # Limpiar placeholders no sustituidos
        import re
        codigo_limpio = re.sub(r'\{[a-z_]+\}', 'x', codigo)
        
        variaciones.append({
            "id": f"{plantilla_key}_{i:03d}",
            "codigo": codigo_limpio,
            "big_o": plantilla["big_o"],
            "patron": plantilla["patron"],
            "plantilla_origen": plantilla_key,
            "nombre_algoritmo": nombre
        })
    
    return variaciones

# Generar variaciones para cada plantilla
todas_las_variaciones = []
for plantilla_key in PLANTILLAS:
    variaciones = generar_variaciones_manual(plantilla_key, n_variaciones=5)
    todas_las_variaciones.extend(variaciones)
    print(f"Plantilla '{plantilla_key}': {len(variaciones)} variaciones generadas")

print(f"\nTotal de variaciones generadas: {len(todas_las_variaciones)}")

---

## 5. Generación por Complejidad Objetivo

In [ ]:
COMPLEJIDADES_OBJETIVO = {
    "O(1)": ["""
algorithm acceso_directo(A[], i)
begin
    return A[i]
end
""", """
algorithm constante(x, y)
begin
    return x + y
end
"""],
    "O(log n)": ["""
algorithm busquedaBinaria(A[], low, high, key)
begin
    while (low <= high) do
        mid <- floor((low + high) / 2)
        if (A[mid] = key) then
            return mid
        end
        if (A[mid] < key) then
            low <- mid + 1
        else
            high <- mid - 1
        end
    end
    return -1
end
"""],
    "O(n)": ["""
algorithm suma_elementos(A[], n)
begin
    total <- 0
    for i <- 1 to n do
        total <- total + A[i]
    end
    return total
end
"""],
    "O(n^2)": ["""
algorithm selectionSort(A[], n)
begin
    for i <- 1 to n - 1 do
        min_idx <- i
        for j <- i + 1 to n do
            if (A[j] < A[min_idx]) then
                min_idx <- j
            end
        end
        temp <- A[i]
        A[i] <- A[min_idx]
        A[min_idx] <- temp
    end
end
"""],
}

# Validar que todos los algoritmos por complejidad se pueden parsear
print("VALIDANDO ALGORITMOS POR COMPLEJIDAD OBJETIVO:")
dataset_por_complejidad = []
for complejidad, codigos in COMPLEJIDADES_OBJETIVO.items():
    validos = 0
    for j, codigo in enumerate(codigos):
        try:
            ast = parse_pseudocode(codigo)
            validos += 1
            dataset_por_complejidad.append({
                "big_o": complejidad,
                "codigo": codigo,
                "nombre": ast.algorithm.name
            })
        except Exception as e:
            print(f"  FAIL {complejidad}[{j}]: {str(e)[:50]}")
    print(f"  {complejidad}: {validos}/{len(codigos)} algoritmos válidos")

---

## 6. Validación de Algoritmos Generados

In [ ]:
engine = AnalyzerEngine()

def validar_algoritmo_generado(entrada):
    """
    Valida que un algoritmo generado sintéticamente:
    1. Se puede parsear
    2. El analizador puede calcular su complejidad
    3. La complejidad calculada coincide con la esperada
    """
    resultado = {
        "id": entrada.get("id", "?"),
        "nombre": entrada.get("nombre_algoritmo", "?"),
        "parsing_ok": False,
        "analisis_ok": False,
        "complejidad_esperada": entrada.get("big_o"),
        "complejidad_obtenida": None,
        "coincide": False
    }
    
    try:
        ast = parse_pseudocode(entrada["codigo"])
        resultado["parsing_ok"] = True
        
        analysis = engine.analyze(ast)
        big_o = (getattr(analysis, 'big_o', None) or
                 getattr(analysis, 'time_complexity', {}).get('big_o'))
        resultado["complejidad_obtenida"] = str(big_o) if big_o else None
        resultado["analisis_ok"] = big_o is not None
        resultado["coincide"] = str(big_o) == str(entrada.get("big_o"))
        
    except Exception as e:
        resultado["error"] = str(e)[:60]
    
    return resultado

print("VALIDANDO VARIACIONES GENERADAS:")
resultados_validacion = [validar_algoritmo_generado(v) for v in todas_las_variaciones[:10]]

validos = sum(1 for r in resultados_validacion if r["parsing_ok"] and r["analisis_ok"])
coincidentes = sum(1 for r in resultados_validacion if r["coincide"])

print(f"Parsing exitoso:          {sum(1 for r in resultados_validacion if r['parsing_ok'])}/{len(resultados_validacion)}")
print(f"Análisis exitoso:         {validos}/{len(resultados_validacion)}")
print(f"Complejidad correcta:     {coincidentes}/{len(resultados_validacion)}")

---

## 7. Estadísticas del Dataset Generado

In [ ]:
from collections import Counter

# Estadísticas del conjunto completo
all_data = todas_las_variaciones + dataset_por_complejidad

print("ESTADÍSTICAS DEL DATASET GENERADO:")
print(f"Total de ejemplos: {len(all_data)}")

# Por complejidad
big_o_counter = Counter(d.get("big_o") for d in all_data)
print(f"\nDistribución por complejidad:")
for complejidad, cantidad in sorted(big_o_counter.items()):
    barra = "#" * (cantidad * 3)
    print(f"  {str(complejidad):<12}: {barra} ({cantidad})")

# Por patrón
patron_counter = Counter(d.get("patron") for d in all_data if d.get("patron"))
print(f"\nDistribución por patrón algorítmico:")
for patron, cantidad in sorted(patron_counter.items(), key=lambda x: -x[1]):
    print(f"  {str(patron):<20}: {cantidad}")


---

## 8. Exportación

In [ ]:
output_dir = Path("../../data/datasets")
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "synthetic_algorithms.json"

dataset_export = {
    "metadata": {
        "total_ejemplos": len(all_data),
        "plantillas_usadas": list(PLANTILLAS.keys()),
        "distribucion_big_o": dict(big_o_counter)
    },
    "ejemplos": all_data
}

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(dataset_export, f, indent=2, ensure_ascii=False)

print(f"Dataset exportado: {output_file}")
print(f"Total de ejemplos guardados: {len(all_data)}")

---

## Proximos Pasos

- **labeling_process.ipynb**: Proceso de etiquetado automático de complejidades
- **dataset_validation.ipynb**: Validación y análisis de calidad del dataset completo